###1) Sprawdzenie metadanych kanałów

In [0]:
import json
from pathlib import Path

METADATA_PATH = Path(
    "/Volumes/workspace/energy_bronze/landing/"
    "nlr/system_2107/metadata/2107_system_metadata.json"
)

with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

metrics_text = json.dumps(
    metadata["Metrics"],
    indent=2,
    ensure_ascii=False
)

CHANNEL_IDS = [
    "149574",  # irradiance
    "149575",  # temperatura
    "149576",  # prędkość wiatru
    "149577",  # kierunek wiatru
    "149578",  # produkcja AC
]

for channel_id in CHANNEL_IDS:
    position = metrics_text.find(channel_id)

    print("\n" + "=" * 70)
    print(f"KANAŁ {channel_id}")
    print("=" * 70)

    if position == -1:
        print("Nie znaleziono kanału w sekcji Metrics")
    else:
        start = max(0, position - 350)
        end = min(len(metrics_text), position + 750)
        print(metrics_text[start:end])

###2) Profil wartości liczbowych

In [0]:
from pyspark.sql import functions as F

MEASUREMENTS = [
    (
        "ac_output",
        "workspace.energy_bronze.nlr_meter_raw",
        "meter_revenue_grade_ac_output_meter_149578"
    ),
    (
        "ambient_temperature",
        "workspace.energy_bronze.nlr_environment_raw",
        "ambient_temperature_o_149575"
    ),
    (
        "wind_speed",
        "workspace.energy_bronze.nlr_environment_raw",
        "wind_speed_o_149576"
    ),
    (
        "wind_direction",
        "workspace.energy_bronze.nlr_environment_raw",
        "wind_direction_o_149577"
    ),
    (
        "poa_irradiance",
        "workspace.energy_bronze.nlr_irradiance_raw",
        "poa_irradiance_o_149574"
    ),
]

profile_rows = []

for measurement_name, table_name, column_name in MEASUREMENTS:
    dataframe = spark.table(table_name)

    raw_value = F.col(column_name)
    numeric_value = F.expr(
        f"try_cast(`{column_name}` AS DOUBLE)"
    )

    nonempty = (
        raw_value.isNotNull()
        & (F.trim(raw_value) != "")
    )

    clean_numeric = F.when(
        F.isnan(numeric_value),
        F.lit(None).cast("double")
    ).otherwise(numeric_value)

    result = (
        dataframe
        .agg(
            F.count("*").alias("row_count"),
            F.sum(
                F.when(~nonempty, 1).otherwise(0)
            ).alias("missing_count"),
            F.sum(
                F.when(
                    nonempty & numeric_value.isNull(),
                    1
                ).otherwise(0)
            ).alias("invalid_numeric_count"),
            F.sum(
                F.when(
                    F.isnan(numeric_value),
                    1
                ).otherwise(0)
            ).alias("nan_count"),
            F.sum(
                F.when(
                    clean_numeric < 0,
                    1
                ).otherwise(0)
            ).alias("negative_count"),
            F.min(clean_numeric).alias("minimum"),
            F.max(clean_numeric).alias("maximum"),
            F.percentile_approx(
                clean_numeric,
                [0.01, 0.50, 0.99],
                10000
            ).alias("percentiles")
        )
        .first()
    )

    percentiles = result["percentiles"]

    profile_rows.append({
        "measurement": measurement_name,
        "row_count": result["row_count"],
        "missing_count": result["missing_count"],
        "invalid_numeric_count": result["invalid_numeric_count"],
        "nan_count": result["nan_count"],
        "negative_count": result["negative_count"],
        "minimum": result["minimum"],
        "p01": percentiles[0] if percentiles else None,
        "median": percentiles[1] if percentiles else None,
        "p99": percentiles[2] if percentiles else None,
        "maximum": result["maximum"],
    })

measurement_profile_df = spark.createDataFrame(profile_rows)

display(measurement_profile_df)

###3) Konfiguracja czasu i reguł jakości

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

LOCAL_TIME_ZONE = "America/Los_Angeles"

SYSTEM_ID = 2107
SYSTEM_DC_CAPACITY_KW = 893.0

# Konserwatywny próg; wartości są tylko flagowane, nie usuwane.
MAX_REASONABLE_AC_POWER_KW = SYSTEM_DC_CAPACITY_KW * 1.10

MIN_TEMPERATURE_F = -40.0
MAX_TEMPERATURE_F = 140.0
MAX_WIND_SPEED_MPH = 150.0
MAX_POA_IRRADIANCE_WM2 = 1500.0

###4) Silver dla produkcji energii

In [0]:
meter_raw = spark.table(
    "workspace.energy_bronze.nlr_meter_raw"
)

meter_local_timestamp = F.to_timestamp(
    F.col("measured_on"),
    "yyyy-MM-dd HH:mm:ss"
)

meter_utc_timestamp = F.to_utc_timestamp(
    meter_local_timestamp,
    LOCAL_TIME_ZONE
)

ac_power_raw = F.expr(
    """
    try_cast(
        `meter_revenue_grade_ac_output_meter_149578`
        AS DOUBLE
    )
    """
)

ac_power_outlier = (
    ac_power_raw.isNotNull()
    & (
        (ac_power_raw < 0)
        | (ac_power_raw > MAX_REASONABLE_AC_POWER_KW)
    )
)

meter_silver = meter_raw.select(
    F.lit(SYSTEM_ID).alias("system_id"),
    F.col("measured_on").alias("measured_on_raw"),
    meter_local_timestamp.alias("measured_at_local"),
    meter_utc_timestamp.alias("measured_at_utc"),
    F.to_date(meter_local_timestamp).alias(
        "measurement_date_local"
    ),
    ac_power_raw.alias("ac_power_kw_raw"),
    F.when(
        ~ac_power_outlier,
        ac_power_raw
    ).alias("ac_power_kw"),
    ac_power_raw.isNull().alias("is_ac_power_missing"),
    ac_power_outlier.alias("is_ac_power_outlier"),
    F.when(
        meter_local_timestamp.isNull(),
        F.lit("INVALID_TIMESTAMP")
    ).when(
        ac_power_raw.isNull(),
        F.lit("MISSING_VALUE")
    ).when(
        ac_power_outlier,
        F.lit("PHYSICAL_OUTLIER")
    ).otherwise(
        F.lit("VALID")
    ).alias("quality_status"),
    "_source_file_name",
    "_record_hash",
    "_ingestion_run_id",
    "_ingested_at_utc"
)

display(meter_silver.limit(10))

###5) Silver dla pogody

In [0]:
environment_raw = spark.table(
    "workspace.energy_bronze.nlr_environment_raw"
)

environment_local_timestamp = F.to_timestamp(
    F.col("measured_on"),
    "yyyy-MM-dd HH:mm:ss"
)

environment_utc_timestamp = F.to_utc_timestamp(
    environment_local_timestamp,
    LOCAL_TIME_ZONE
)

temperature_f = F.expr(
    "try_cast(`ambient_temperature_o_149575` AS DOUBLE)"
)

wind_speed_mph = F.expr(
    "try_cast(`wind_speed_o_149576` AS DOUBLE)"
)

wind_direction_raw = F.expr(
    "try_cast(`wind_direction_o_149577` AS DOUBLE)"
)

temperature_outlier = (
    temperature_f.isNotNull()
    & ~temperature_f.between(
        MIN_TEMPERATURE_F,
        MAX_TEMPERATURE_F
    )
)

wind_speed_outlier = (
    wind_speed_mph.isNotNull()
    & ~wind_speed_mph.between(
        0.0,
        MAX_WIND_SPEED_MPH
    )
)

wind_direction_outlier = (
    wind_direction_raw.isNotNull()
    & ~wind_direction_raw.between(0.0, 360.0)
)

is_any_weather_outlier = (
    temperature_outlier
    | wind_speed_outlier
    | wind_direction_outlier
)

environment_silver = environment_raw.select(
    F.lit(SYSTEM_ID).alias("system_id"),
    F.col("measured_on").alias("measured_on_raw"),
    environment_local_timestamp.alias("measured_at_local"),
    environment_utc_timestamp.alias("measured_at_utc"),
    F.to_date(environment_local_timestamp).alias(
        "measurement_date_local"
    ),

    temperature_f.alias("ambient_temperature_f_raw"),
    F.when(
        ~temperature_outlier,
        (temperature_f - F.lit(32.0)) * F.lit(5.0 / 9.0)
    ).alias("ambient_temperature_c"),

    wind_speed_mph.alias("wind_speed_mph_raw"),
    F.when(
        ~wind_speed_outlier,
        wind_speed_mph * F.lit(0.44704)
    ).alias("wind_speed_ms"),

    wind_direction_raw.alias("wind_direction_deg_raw"),
    F.when(
        ~wind_direction_outlier,
        F.pmod(wind_direction_raw, F.lit(360.0))
    ).alias("wind_direction_deg"),

    temperature_f.isNull().alias("is_temperature_missing"),
    wind_speed_mph.isNull().alias("is_wind_speed_missing"),
    wind_direction_raw.isNull().alias(
        "is_wind_direction_missing"
    ),

    temperature_outlier.alias("is_temperature_outlier"),
    wind_speed_outlier.alias("is_wind_speed_outlier"),
    wind_direction_outlier.alias(
        "is_wind_direction_outlier"
    ),
    is_any_weather_outlier.alias(
        "is_any_weather_outlier"
    ),

    "_source_file_name",
    "_record_hash",
    "_ingestion_run_id",
    "_ingested_at_utc"
)

display(environment_silver.limit(10))

###6) Silver dla irradiacji

In [0]:
irradiance_raw = spark.table(
    "workspace.energy_bronze.nlr_irradiance_raw"
)

irradiance_local_timestamp = F.to_timestamp(
    F.col("measured_on"),
    "yyyy-MM-dd HH:mm:ss"
)

irradiance_utc_timestamp = F.to_utc_timestamp(
    irradiance_local_timestamp,
    LOCAL_TIME_ZONE
)

poa_irradiance_raw = F.expr(
    "try_cast(`poa_irradiance_o_149574` AS DOUBLE)"
)

irradiance_outlier = (
    poa_irradiance_raw.isNotNull()
    & ~poa_irradiance_raw.between(
        0.0,
        MAX_POA_IRRADIANCE_WM2
    )
)

irradiance_silver = irradiance_raw.select(
    F.lit(SYSTEM_ID).alias("system_id"),
    F.col("measured_on").alias("measured_on_raw"),
    irradiance_local_timestamp.alias("measured_at_local"),
    irradiance_utc_timestamp.alias("measured_at_utc"),
    F.to_date(irradiance_local_timestamp).alias(
        "measurement_date_local"
    ),
    poa_irradiance_raw.alias("poa_irradiance_wm2_raw"),
    F.when(
        ~irradiance_outlier,
        poa_irradiance_raw
    ).alias("poa_irradiance_wm2"),
    poa_irradiance_raw.isNull().alias(
        "is_irradiance_missing"
    ),
    irradiance_outlier.alias(
        "is_irradiance_outlier"
    ),
    "_source_file_name",
    "_record_hash",
    "_ingestion_run_id",
    "_ingested_at_utc"
)

display(irradiance_silver.limit(10))

###7) Zapis tabel Delta

In [0]:
SILVER_TABLES = {
    "workspace.energy_silver.nlr_meter": meter_silver,
    "workspace.energy_silver.nlr_environment": environment_silver,
    "workspace.energy_silver.nlr_irradiance": irradiance_silver,
}

for table_name, dataframe in SILVER_TABLES.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Zapisano: {table_name}")

###8) Kontrola Silver

In [0]:
QUALITY_CONFIG = [
    (
        "meter",
        meter_silver,
        "is_ac_power_outlier"
    ),
    (
        "environment",
        environment_silver,
        "is_any_weather_outlier"
    ),
    (
        "irradiance",
        irradiance_silver,
        "is_irradiance_outlier"
    ),
]

silver_profile = []

for dataset_name, dataframe, quality_column in QUALITY_CONFIG:
    result = dataframe.agg(
        F.count("*").alias("row_count"),
        F.min("measured_at_utc").alias("min_utc"),
        F.max("measured_at_utc").alias("max_utc"),
        F.sum(
            F.when(
                F.col("measured_at_utc").isNull(),
                1
            ).otherwise(0)
        ).alias("invalid_timestamp_count"),
        F.sum(
            F.when(
                F.col(quality_column),
                1
            ).otherwise(0)
        ).alias("physical_outlier_count")
    ).first()

    silver_profile.append({
        "dataset": dataset_name,
        "row_count": result["row_count"],
        "min_utc": result["min_utc"],
        "max_utc": result["max_utc"],
        "invalid_timestamp_count": (
            result["invalid_timestamp_count"]
        ),
        "physical_outlier_count": (
            result["physical_outlier_count"]
        ),
    })

display(spark.createDataFrame(silver_profile))

###9) Kontrola powtarzających się timestampów

In [0]:
duplicate_results = []

for dataset_name, dataframe, _ in QUALITY_CONFIG:
    duplicate_groups = (
        dataframe
        .where(F.col("measured_at_utc").isNotNull())
        .groupBy("measured_at_utc")
        .count()
        .where(F.col("count") > 1)
    )

    result = duplicate_groups.agg(
        F.count("*").alias("duplicate_timestamp_groups"),
        F.sum(
            F.col("count") - F.lit(1)
        ).alias("additional_rows")
    ).first()

    duplicate_results.append({
        "dataset": dataset_name,
        "duplicate_timestamp_groups": (
            result["duplicate_timestamp_groups"]
        ),
        "additional_rows": result["additional_rows"] or 0,
    })

display(spark.createDataFrame(duplicate_results))

###10) Podejrzane skoki mocy

In [0]:
display(
    meter_silver
    .where(F.col("is_ac_power_outlier"))
    .select(
        "measured_on_raw",
        "measured_at_utc",
        "ac_power_kw_raw",
        "quality_status",
        "_source_file_name"
    )
    .orderBy(F.desc("ac_power_kw_raw"))
    .limit(20)
)

###11)

In [0]:
import json
from pathlib import Path

METADATA_PATH = Path(
    "/Volumes/workspace/energy_bronze/landing/"
    "nlr/system_2107/metadata/2107_system_metadata.json"
)

with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

for section_name in ["Meters", "Other Instruments"]:
    print("\n" + "=" * 80)
    print(section_name.upper())
    print("=" * 80)

    print(
        json.dumps(
            metadata.get(section_name, {}),
            indent=2,
            ensure_ascii=False
        )
    )

###12) Interwały mocy i irradiancji

In [0]:
from pyspark.sql import functions as F

meter_silver = spark.table(
    "workspace.energy_silver.nlr_meter"
)

environment_silver = spark.table(
    "workspace.energy_silver.nlr_environment"
)

irradiance_silver = spark.table(
    "workspace.energy_silver.nlr_irradiance"
)

# Moc jest średnią dla lewostronnie oznaczonego okresu 15 minut.
meter_intervals = (
    meter_silver
    .where(F.col("measured_at_utc").isNotNull())
    .select(
        "system_id",
        F.col("measured_at_utc").alias(
            "interval_start_utc"
        ),
        (
            F.col("measured_at_utc")
            + F.expr("INTERVAL 15 MINUTES")
        ).alias("interval_end_utc"),
        F.col("ac_power_kw_raw"),
        F.col("ac_power_kw"),
        (
            F.col("ac_power_kw") * F.lit(0.25 / 1000.0)
        ).alias("pv_energy_mwh"),
        F.col("is_ac_power_missing"),
        F.col("is_ac_power_outlier"),
        F.col("quality_status")
    )
)

# Irradiancja ma większą częstotliwość, dlatego agregujemy ją
# do lewostronnych przedziałów 15-minutowych.
irradiance_15m = (
    irradiance_silver
    .where(F.col("measured_at_utc").isNotNull())
    .groupBy(
        "system_id",
        F.window(
            "measured_at_utc",
            "15 minutes"
        ).alias("interval_15m")
    )
    .agg(
        F.avg("poa_irradiance_wm2").alias(
            "poa_irradiance_avg_wm2"
        ),
        F.count("poa_irradiance_wm2").alias(
            "poa_valid_sample_count"
        ),
        F.count("*").alias(
            "poa_total_sample_count"
        ),
        F.max(
            F.col("is_irradiance_outlier").cast("integer")
        ).cast("boolean").alias(
            "has_irradiance_outlier"
        )
    )
    .select(
        "system_id",
        F.col("interval_15m.start").alias(
            "interval_start_utc"
        ),
        F.col("interval_15m.end").alias(
            "interval_end_utc"
        ),
        "poa_irradiance_avg_wm2",
        "poa_valid_sample_count",
        "poa_total_sample_count",
        "has_irradiance_outlier"
    )
)

display(
    irradiance_15m
    .groupBy("poa_total_sample_count")
    .count()
    .orderBy("poa_total_sample_count")
)

###13) Utworzenie momentów decyzyjnych

In [0]:
# Ostatnia zakończona produkcja, dostępna w momencie decyzji.
meter_previous = meter_intervals.select(
    "system_id",
    F.col("interval_end_utc").alias(
        "decision_time_utc"
    ),
    F.col("interval_start_utc").alias(
        "previous_interval_start_utc"
    ),
    F.col("interval_end_utc").alias(
        "power_available_at_utc"
    ),
    F.col("ac_power_kw_raw").alias(
        "ac_power_previous_15m_kw_raw"
    ),
    F.col("ac_power_kw").alias(
        "ac_power_previous_15m_kw"
    ),
    F.col("pv_energy_mwh").alias(
        "pv_energy_previous_15m_mwh"
    ),
    F.col("is_ac_power_outlier").alias(
        "previous_power_is_outlier"
    )
)

# Irradiancja z zakończonego okresu, dostępna w momencie decyzji.
irradiance_previous = irradiance_15m.select(
    "system_id",
    F.col("interval_end_utc").alias(
        "decision_time_utc"
    ),
    F.col("interval_start_utc").alias(
        "poa_interval_start_utc"
    ),
    F.col("interval_end_utc").alias(
        "poa_available_at_utc"
    ),
    F.col("poa_irradiance_avg_wm2").alias(
        "poa_previous_15m_wm2"
    ),
    "poa_valid_sample_count",
    "poa_total_sample_count",
    "has_irradiance_outlier"
)

# Bieżąca obserwacja pogodowa. Metryki pogodowe mają typ "sample",
# więc obserwacja z czasu t może być użyta przy decyzji w czasie t.
environment_at_decision = environment_silver.select(
    "system_id",
    F.col("measured_at_utc").alias(
        "decision_time_utc"
    ),
    F.col("measured_at_utc").alias(
        "weather_observed_at_utc"
    ),
    F.col("ambient_temperature_c").alias(
        "ambient_temperature_c"
    ),
    F.col("wind_speed_ms").alias(
        "wind_speed_ms"
    ),
    F.col("wind_direction_deg").alias(
        "wind_direction_deg"
    ),
    F.col("is_temperature_missing"),
    F.col("is_wind_speed_missing"),
    F.col("is_wind_direction_missing"),
    F.col("is_any_weather_outlier")
)

# Produkcja w następnym przedziale. Jest to wyłącznie etykieta
# do oceny symulacji, a nie cecha dostępna dla modelu.
meter_target = meter_intervals.select(
    "system_id",
    F.col("interval_start_utc").alias(
        "decision_time_utc"
    ),
    F.col("interval_start_utc").alias(
        "target_interval_start_utc"
    ),
    F.col("interval_end_utc").alias(
        "target_interval_end_utc"
    ),
    F.col("ac_power_kw_raw").alias(
        "actual_ac_power_next_15m_kw_raw"
    ),
    F.col("ac_power_kw").alias(
        "actual_ac_power_next_15m_kw"
    ),
    F.col("pv_energy_mwh").alias(
        "actual_pv_energy_next_15m_mwh"
    ),
    F.col("is_ac_power_outlier").alias(
        "target_power_is_outlier"
    )
)

###14) Połączenie tabel

In [0]:
nlr_decision_features_15m = (
    meter_previous
    .join(
        irradiance_previous,
        ["system_id", "decision_time_utc"],
        "left"
    )
    .join(
        environment_at_decision,
        ["system_id", "decision_time_utc"],
        "left"
    )
    .join(
        meter_target,
        ["system_id", "decision_time_utc"],
        "left"
    )
    .withColumn(
        "decision_date_utc",
        F.to_date("decision_time_utc")
    )
    .withColumn(
        "is_core_feature_complete",
        F.col("ac_power_previous_15m_kw").isNotNull()
        & F.col("poa_previous_15m_wm2").isNotNull()
        & F.col("ambient_temperature_c").isNotNull()
    )
    .withColumn(
        "has_next_interval_target",
        F.col("actual_ac_power_next_15m_kw").isNotNull()
    )
)

display(
    nlr_decision_features_15m
    .orderBy("decision_time_utc")
    .limit(20)
)

###15) Zapis tabeli

In [0]:
(
    nlr_decision_features_15m.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.energy_silver.nlr_decision_features_15m"
    )
)

print(
    "Zapisano: "
    "workspace.energy_silver.nlr_decision_features_15m"
)

###16) Kontrola kompletności

In [0]:
decision_summary = (
    nlr_decision_features_15m
    .agg(
        F.count("*").alias("decision_row_count"),
        F.min("decision_time_utc").alias(
            "min_decision_time_utc"
        ),
        F.max("decision_time_utc").alias(
            "max_decision_time_utc"
        ),
        F.sum(
            F.when(
                F.col("ac_power_previous_15m_kw").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_previous_power"),
        F.sum(
            F.when(
                F.col("poa_previous_15m_wm2").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_poa"),
        F.sum(
            F.when(
                F.col("ambient_temperature_c").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_temperature"),
        F.sum(
            F.when(
                F.col("wind_speed_ms").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_wind_speed"),
        F.sum(
            F.when(
                ~F.col("has_next_interval_target"),
                1
            ).otherwise(0)
        ).alias("missing_next_interval_target"),
        F.sum(
            F.when(
                F.col("is_core_feature_complete"),
                1
            ).otherwise(0)
        ).alias("complete_core_feature_rows")
    )
)

display(decision_summary)

###17) Test braku wycieku przyszłości

In [0]:
leakage_violations = (
    nlr_decision_features_15m
    .where(
        (
            F.col("power_available_at_utc")
            > F.col("decision_time_utc")
        )
        |
        (
            F.col("poa_available_at_utc")
            > F.col("decision_time_utc")
        )
        |
        (
            F.col("weather_observed_at_utc")
            > F.col("decision_time_utc")
        )
    )
    .count()
)

assert leakage_violations == 0, (
    f"Wykryto {leakage_violations} przypadków data leakage"
)

print("Test zaliczony: brak wykorzystania przyszłych obserwacji.")